Installing Sentence Transformer Package

In [1]:
!pip install -U sentence-transformers

  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 3.0.1
    Uninstalling sentence-transformers-3.0.1:
      Successfully uninstalled sentence-transformers-3.0.1


In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
# Load the SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading Recipes

In [20]:
# Load recipe data from Excel
file_path = "merged_recipes.xlsx"  # Update the file name if needed
recipe_df = pd.read_excel(file_path)

# Display the first few rows to confirm
print(recipe_df.head())

           Recipe Name                                        Ingredients  \
0                Pulao  Rice, Mutton/Beef, Whole Spices, Onions, Ginge...   
1               Tahari  Rice, Potatoes, Onions, Tomatoes, Green Chilie...   
2          Saag Chawal  Rice, Spinach, Mustard Greens, Green Chilies, ...   
3         Machli Pulao  Rice, Fish, Yogurt, Tomatoes, Onions, Green Ch...   
4  Chicken Qorma Pulao  Rice, Chicken, Yogurt, Qorma Masala, Onions, G...   

  Cuisine Region      Cooking Method  \
0         Punjab  Boiling, Simmering   
1         Punjab     One-Pot Cooking   
2         Punjab   Boiling, Sautéing   
3          Sindh  Boiling, Simmering   
4         Punjab  Boiling, Simmering   

                                              Recipe Category  
0  Sauté onions and whole spices in ghee. Add mea...     Rice  
1  Sauté onions and spices in ghee. Add potatoes ...     Rice  
2  Boil rice separately. Sauté spinach and mustar...     Rice  
3  Cook fish with spices and tomatoes. B

In [21]:
# Combine columns into a single string for each recipe
recipe_df['combined_text'] = recipe_df['Ingredients'] + " | " + recipe_df['Cuisine Region'] + " | " + recipe_df['Cooking Method'] + " | " + recipe_df['Recipe']

# Convert the combined text to a list
recipes = recipe_df['combined_text'].tolist()


Converting Recipe Description in Embeddings

In [22]:
# Encode all recipes into embeddings
recipe_embeddings = model.encode(recipes)
print(recipe_embeddings)

[[-0.06713837 -0.01075819 -0.0261787  ... -0.01019946 -0.02555402
   0.0402103 ]
 [-0.08712653 -0.02991522 -0.05945076 ...  0.02111084  0.00881603
   0.04504044]
 [-0.05902212  0.00561172 -0.06268251 ... -0.0059495  -0.01464393
   0.04271801]
 ...
 [-0.05965048 -0.02368429 -0.06712977 ... -0.02076235  0.00541555
   0.01067809]
 [-0.01770667 -0.07450178  0.04579772 ...  0.04187557  0.12196068
   0.04115935]
 [-0.01731122  0.02825663 -0.01001586 ...  0.01385877 -0.06634368
   0.06227247]]


In [29]:
print(recipe_embeddings.shape)  # Should be (number_of_recipes, embedding_dimension)
print(recipe_embeddings[0])     # View the embedding for the first recipe


(3305, 384)
[-6.71383664e-02 -1.07581895e-02 -2.61787027e-02  7.57943913e-02
 -1.14311211e-01  5.36425337e-02 -3.14057246e-02 -8.69567096e-02
  5.91023732e-03 -6.39985576e-02  9.90367457e-02 -7.82408565e-02
 -1.35887237e-02 -5.35124950e-02  4.02864031e-02 -5.13305925e-02
  1.15933239e-01  4.76292893e-03 -2.12602429e-02 -1.56576574e-01
 -4.15239558e-02 -4.08815360e-03  2.33754013e-02 -3.56671736e-02
 -1.29412897e-02 -3.00891027e-02  4.38455716e-02  2.06905305e-02
 -1.78965572e-02 -4.27860469e-02  4.23068665e-02 -2.78890859e-02
  3.29848044e-02  4.96374555e-02 -2.88819745e-02  1.19067812e-02
 -2.01742891e-02 -4.15171906e-02  7.75361061e-02  7.01771230e-02
  8.86630490e-02  5.98718897e-02  5.29693589e-02  7.68508250e-03
  5.86240441e-02  1.86700989e-02 -3.84730957e-02 -8.74107424e-03
  6.85224123e-03 -5.89784794e-02 -3.49703282e-02 -2.61038151e-02
 -4.33076210e-02 -2.51116673e-03  6.82718083e-02 -8.66372064e-02
 -5.89726865e-02 -3.78785692e-02  3.01038921e-02  5.36629930e-02
 -1.84411202e

In [33]:
# User input for matching
user_input = "Rice, Potatoes, Onions"

# Generate embedding for user input
user_embedding = model.encode([user_input])


In [34]:
# Compute similarity scores
similarities = cosine_similarity(user_embedding, recipe_embeddings)

# Find the most similar recipe
most_similar_index = similarities[0].argmax()
print("Best Match Recipe Name:", recipe_df.iloc[most_similar_index]['Recipe Name'])

# Display top-N matches
top_n = 3
top_indices = similarities[0].argsort()[-top_n:][::-1]
print("\nTop Matches:")
for idx in top_indices:
    print(f"{recipe_df.iloc[idx]['Recipe Name']} (Similarity: {similarities[0][idx]:.2f})")


Best Match Recipe Name: Karachi Street Food Rice

Top Matches:
Karachi Street Food Rice (Similarity: 0.68)
Chaat Masala Rice (Similarity: 0.68)
Fried Rice (Similarity: 0.67)


In [35]:
# Compute similarity scores
similarities = cosine_similarity(user_embedding, recipe_embeddings)

# Find the most similar recipe
most_similar_index = similarities[0].argmax()
print("Best Match Recipe Name:", recipe_df.iloc[most_similar_index]['Recipe Name'])

# Display top-N matches
top_n = 3
top_indices = similarities[0].argsort()[-top_n:][::-1]  # Sort to get top N indices in descending order

print("\nTop Matches:")
for idx in top_indices:
    recipe_info = recipe_df.iloc[idx]
    print(f"Recipe Name: {recipe_info['Recipe Name']} (Similarity: {similarities[0][idx]:.2f})")
    print(f"  Ingredients: {recipe_info['Ingredients']}")
    print(f"  Cuisine Region: {recipe_info['Cuisine Region']}")
    print(f"  Cooking Method: {recipe_info['Cooking Method']}")
    print(f"  Recipe: {recipe_info['Recipe']}")
    print(f"  Category: {recipe_info['Category']}\n")


Best Match Recipe Name: Karachi Street Food Rice

Top Matches:
Recipe Name: Karachi Street Food Rice (Similarity: 0.68)
  Ingredients: Rice, Chaat Masala, Potatoes, Onions, Yogurt
  Cuisine Region: Fusion
  Cooking Method: Boiling, Mixing
  Recipe: Mix boiled rice with chaat masala, boiled potatoes, onions, and top with yogurt.
  Category: Rice

Recipe Name: Chaat Masala Rice (Similarity: 0.68)
  Ingredients: Rice, Chaat Masala, Potatoes, Onions, Yogurt
  Cuisine Region: Fusion
  Cooking Method: Boiling, Mixing
  Recipe: Mix boiled rice with chaat masala, boiled potatoes, onions, and top with yogurt.
  Category: Vegetarian

Recipe Name: Fried Rice (Similarity: 0.67)
  Ingredients: Rice, Vegetables (Carrots, Peas), Soy Sauce, Green Chilies, Ghee
  Cuisine Region: Various
  Cooking Method: Stir-Frying
  Recipe: Stir-fry cooked rice with vegetables and soy sauce. Serve hot.
  Category: Rice



In [44]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Precision and Recall at k
def precision_recall_at_k(y_true, y_pred, k):
    y_pred_at_k = y_pred[:k]
    relevant = set(y_true) & set(y_pred_at_k)
    precision = len(relevant) / k
    recall = len(relevant) / len(y_true) if len(y_true) > 0 else 0.0
    return precision, recall

# Mean Average Precision
def mean_average_precision(y_true, y_pred):
    ap = 0.0
    relevant = 0
    for i, pred in enumerate(y_pred):
        if pred in y_true:
            relevant += 1
            ap += relevant / (i + 1)
    return ap / len(y_true) if len(y_true) > 0 else 0.0

# Mean Reciprocal Rank
def mean_reciprocal_rank(y_true, y_pred):
    for i, pred in enumerate(y_pred):
        if pred in y_true:
            return 1 / (i + 1)
    return 0.0

# Example recipe data
# Assuming recipe_df is a DataFrame with columns: Recipe Name, Ingredients, Cuisine Region, Cooking Method, and Category
# Also assuming recipe_embeddings are pre-computed embeddings for all recipes
test_queries = [
    {"user_input": "Rice, Chicken, Spices", "relevant_recipes": ["Chicken Pulao", "Chicken Biryani"]},
    {"user_input": "Flour, Sugar, Butter, Eggs", "relevant_recipes": ["Cake", "Cupcakes", "Butter Cookies"]},
    {"user_input": "Yogurt, Chicken, Spices", "relevant_recipes": ["Chicken Yogurt Curry", "Yogurt Chicken Gravy"]},
    {"user_input": "Lentils, Tomatoes, Spices", "relevant_recipes": ["Lentil Soup", "Dal"]},
    {"user_input": "Potatoes, Chicken, Onions", "relevant_recipes": ["Chicken Masala", "Chicken Potato Curry"]},
    {"user_input": "Milk, Sugar, Rice", "relevant_recipes": ["Kheer", "Rice Pudding"]},
    {"user_input": "Tomatoes, Chicken, Garlic", "relevant_recipes": ["Chicken Curry", "Tomato Chicken"]},
    {"user_input": "Spinach, Paneer, Spices", "relevant_recipes": ["Palak Paneer", "Spinach Curry"]},
    {"user_input": "Flour, Water, Salt, Oil", "relevant_recipes": ["Chapati", "Paratha"]},
    {"user_input": "Bananas, Sugar, Flour", "relevant_recipes": ["Banana Cake", "Banana Bread"]}
]

# Main evaluation loop
overall_precision = []
overall_recall = []
overall_map = []
overall_mrr = []

# Ensure recipe_df and recipe_embeddings match
num_recipes = len(recipe_df)
assert len(recipe_embeddings) == num_recipes, "Mismatch between recipe_df and recipe_embeddings lengths"

for query in test_queries:
    # Generate embedding for the user's input
    user_embedding = model.encode([query["user_input"]])
    
    # Compute similarity scores
    similarities = cosine_similarity(user_embedding, recipe_embeddings)
    
    # Rank recipes by similarity
    ranked_indices = similarities[0].argsort()[::-1]
    ranked_recipes = recipe_df.iloc[ranked_indices[:num_recipes]]["Recipe Name"].tolist()  # Safeguard with slicing
    
    # Map relevant recipe names to indices
    relevant_indices = [ranked_recipes.index(recipe) for recipe in query["relevant_recipes"] if recipe in ranked_recipes]
    
    # Evaluate metrics
    precision, recall = precision_recall_at_k(relevant_indices, ranked_indices, k=3)
    map_score = mean_average_precision(relevant_indices, ranked_indices)
    mrr_score = mean_reciprocal_rank(relevant_indices, ranked_indices)
    
    # Append to overall results
    overall_precision.append(precision)
    overall_recall.append(recall)
    overall_map.append(map_score)
    overall_mrr.append(mrr_score)
    
    # Print results for the current query
    print(f"Query: {query['user_input']}")
    print(f"Top 3 Predictions: {ranked_recipes[:3]}")
    print(f"Precision@3: {precision:.2f}, Recall@3: {recall:.2f}")
    print(f"Mean Average Precision: {map_score:.2f}")
    print(f"Mean Reciprocal Rank: {mrr_score:.2f}\n")

# Print overall metrics
print("Overall Metrics:")
print(f"Mean Precision@3: {np.mean(overall_precision):.2f}")
print(f"Mean Recall@3: {np.mean(overall_recall):.2f}")
print(f"Mean Average Precision: {np.mean(overall_map):.2f}")
print(f"Mean Reciprocal Rank: {np.mean(overall_mrr):.2f}")


AssertionError: Mismatch between recipe_df and recipe_embeddings lengths

In [17]:
def evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3):
    precision_scores = []
    recall_scores = []
    map_scores = []
    mrr_scores = []

    for query in test_queries:
        user_input = query['user_input']
        relevant_recipes = query['relevant_recipes']

        # Generate recommendations
        recommendations, similarities = generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n)

        # Calculate metrics
        precision_scores.append(precision_at_n(recommendations, relevant_recipes, top_n))
        recall_scores.append(recall_at_n(recommendations, relevant_recipes, top_n))
        map_scores.append(mean_average_precision(recommendations, relevant_recipes))
        mrr_scores.append(mean_reciprocal_rank(recommendations, relevant_recipes))

    # Print results
    print(f"Precision@{top_n}: {sum(precision_scores) / len(precision_scores):.2f}")
    print(f"Recall@{top_n}: {sum(recall_scores) / len(recall_scores):.2f}")
    print(f"Mean Average Precision (MAP): {sum(map_scores) / len(map_scores):.2f}")
    print(f"Mean Reciprocal Rank (MRR): {sum(mrr_scores) / len(mrr_scores):.2f}")


In [38]:
import random

# Sample function to generate test queries from the recipe dataframe
def generate_test_queries_from_csv(recipe_df, top_n=3):
    test_queries = []

    # Iterate over a random selection of rows from the dataset
    sample_recipes = recipe_df.sample(n=10)  # Adjust n for more test queries

    for index, row in sample_recipes.iterrows():
        recipe_name = row['Recipe Name']
        ingredients = row['Ingredients']

        # Find other recipes with similar ingredients
        relevant_recipes = recipe_df[recipe_df['Ingredients'].apply(lambda x: any(ingredient in x for ingredient in ingredients.split(', ')))]

        # Ensure that we get multiple relevant recipes (top_n)
        relevant_recipes_list = relevant_recipes['Recipe Name'].tolist()
        relevant_recipes_list = random.sample(relevant_recipes_list, min(top_n, len(relevant_recipes_list)))

        test_queries.append({
            'user_input': ingredients,
            'relevant_recipes': relevant_recipes_list
        })
    
    return test_queries

# Generate test queries from your CSV data
test_queries = generate_test_queries_from_csv(recipe_df, top_n=3)

# Print out the generated test queries
for query in test_queries:
    print(f"User Input: {query['user_input']}")
    print(f"Relevant Recipes: {', '.join(query['relevant_recipes'])}\n")


User Input: Hibiscus tea, ginger, soda
Relevant Recipes: Ginger Lime Cooler, Ginger Apple Spritzer, Beef Kofta Curry

User Input: Chicken, pineapple, soy sauce, ginger, garlic
Relevant Recipes: Chicken and Broccoli Rice, Beef and Zucchini Stir-Fry, BBQ Chicken Tikka Sandwich

User Input: Basmati Rice, Pumpkin, Spices
Relevant Recipes: Green Rice, Stir-Fried Vegetable Rice, Pulao Khichdi

User Input: Chickpeas, onions, tomatoes, spices, chutney
Relevant Recipes: Egg Curry with Rice, Prawn and Avocado Ceviche, Avocado Chickpea Salad

User Input: Mutton, spices, tomatoes
Relevant Recipes: Keema Naan, Spiced Lychee Iced Tea, Grilled Shrimp & Tomato Cucumber Salad

User Input: Wrap, assorted vegetables, sauces
Relevant Recipes: Veggie Wrap Rice, Grilled Paneer & Carrot Sliders, Tandoori Chaat

User Input: Mango juice, green tea, chili powder
Relevant Recipes: Spicy Mint Cooler, Spicy Pomegranate Sparkler, BBQ Paneer Skewers

User Input: Beef rendang, samosa wrappers, spices
Relevant Recipes

In [42]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load recipe data from Excel
file_path = "merged_recipes.xlsx"  # Update the file name if needed
recipe_df = pd.read_excel(file_path)

# Display the first few rows to confirm
print(recipe_df.head())

# Combine columns into a single string for each recipe
recipe_df['combined_text'] = recipe_df['Ingredients'] + " | " + recipe_df['Cuisine Region'] + " | " + recipe_df['Cooking Method'] + " | " + recipe_df['Recipe']

# Convert the combined text to a list
recipes = recipe_df['combined_text'].tolist()

# Encode all recipes into embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Use the SentenceTransformer model of your choice
recipe_embeddings = model.encode(recipes)

# User input for matching
user_input = "Rice, Potatoes, Onions"

# Generate embedding for user input
user_embedding = model.encode([user_input])

# Compute similarity scores
similarities = cosine_similarity(user_embedding, recipe_embeddings)

# Get available categories and cuisine regions
available_categories = recipe_df['Category'].unique()
available_cuisines = recipe_df['Cuisine Region'].unique()

# Display available categories and cuisines
print("\nAvailable Categories:")
for category in available_categories:
    print(f"- {category}")

print("\nAvailable Cuisine Regions:")
for cuisine in available_cuisines:
    print(f"- {cuisine}")

# Ask user if they want to narrow down the search by category or cuisine region
category_choice = input("\nWould you like to narrow the search by category? (yes/no): ").strip().lower()
cuisine_choice = input("Would you like to narrow the search by cuisine region? (yes/no): ").strip().lower()

# Filter based on category if user chooses to
if category_choice == 'yes':
    selected_category = input(f"Please choose a category from the above options: ").strip()
    recipe_df = recipe_df[recipe_df['Category'] == selected_category]

# Filter based on cuisine region if user chooses to
if cuisine_choice == 'yes':
    selected_cuisine = input(f"Please choose a cuisine region from the above options: ").strip()
    recipe_df = recipe_df[recipe_df['Cuisine Region'] == selected_cuisine]

# Combine columns again for the filtered recipes
filtered_recipes = recipe_df['Ingredients'] + " | " + recipe_df['Cuisine Region'] + " | " + recipe_df['Cooking Method'] + " | " + recipe_df['Recipe']
filtered_recipe_embeddings = model.encode(filtered_recipes.tolist())

# Compute similarity scores with the filtered recipes
similarities = cosine_similarity(user_embedding, filtered_recipe_embeddings)

# Find the most similar recipe
most_similar_index = similarities[0].argmax()
print("\nBest Match Recipe Name:", recipe_df.iloc[most_similar_index]['Recipe Name'])

# Display top-N matches
top_n = 3
top_indices = similarities[0].argsort()[-top_n:][::-1]  # Sort to get top N indices in descending order

print("\nTop Matches:")
for idx in top_indices:
    recipe_info = recipe_df.iloc[idx]
    print(f"Recipe Name: {recipe_info['Recipe Name']} (Similarity: {similarities[0][idx]:.2f})")
    print(f"  Ingredients: {recipe_info['Ingredients']}")
    print(f"  Cuisine Region: {recipe_info['Cuisine Region']}")
    print(f"  Cooking Method: {recipe_info['Cooking Method']}")
    print(f"  Recipe: {recipe_info['Recipe']}")
    print(f"  Category: {recipe_info['Category']}\n")


           Recipe Name                                        Ingredients  \
0                Pulao  Rice, Mutton/Beef, Whole Spices, Onions, Ginge...   
1               Tahari  Rice, Potatoes, Onions, Tomatoes, Green Chilie...   
2          Saag Chawal  Rice, Spinach, Mustard Greens, Green Chilies, ...   
3         Machli Pulao  Rice, Fish, Yogurt, Tomatoes, Onions, Green Ch...   
4  Chicken Qorma Pulao  Rice, Chicken, Yogurt, Qorma Masala, Onions, G...   

  Cuisine Region      Cooking Method  \
0         Punjab  Boiling, Simmering   
1         Punjab     One-Pot Cooking   
2         Punjab   Boiling, Sautéing   
3          Sindh  Boiling, Simmering   
4         Punjab  Boiling, Simmering   

                                              Recipe Category  
0  Sauté onions and whole spices in ghee. Add mea...     Rice  
1  Sauté onions and spices in ghee. Add potatoes ...     Rice  
2  Boil rice separately. Sauté spinach and mustar...     Rice  
3  Cook fish with spices and tomatoes. B


Would you like to narrow the search by category? (yes/no):  YES
Would you like to narrow the search by cuisine region? (yes/no):  yes
Please choose a category from the above options:  Rice
Please choose a cuisine region from the above options:  Sindh



Best Match Recipe Name: Karachi Tikka Biryani

Top Matches:
Recipe Name: Karachi Tikka Biryani (Similarity: 0.51)
  Ingredients: Rice, Chicken Tikka, Biryani Masala, Potatoes, Onions
  Cuisine Region: Sindh
  Cooking Method: Boiling, Layering, Dum
  Recipe: Cook chicken tikka and layer with rice. Cook on dum. Serve with raita and salad.
  Category: Rice

Recipe Name: Hara Masala Biryani (Similarity: 0.50)
  Ingredients: Rice, Chicken, Green Masala (Coriander, Mint, Green Chilies), Yogurt, Biryani Masala, Onions, Ghee
  Cuisine Region: Sindh
  Cooking Method: Boiling, Layering, Dum
  Recipe: Marinate chicken with green masala and yogurt. Cook with onions and biryani masala. Layer with rice and ghee, cook on dum. Serve with raita and salad.
  Category: Rice

Recipe Name: Sindhi Pulao (Similarity: 0.49)
  Ingredients: Rice, Chicken, Tomatoes, Onions, Green Chilies, Spices
  Cuisine Region: Sindh
  Cooking Method: Boiling, Layering, Dum
  Recipe: Cook chicken with tomatoes and green chili